# 학습된 행동 분류 모델로 test.png 예측

`test/test.png` 한 장에 대해 예측 클래스와 클래스별 확률을 출력합니다.

In [ ]:
from pathlib import Path

from ultralytics import YOLO

# 노트북 cwd가 notebooks/ 이거나 ai-server/ 일 때 모두 동작
ROOT = Path.cwd()
if not (ROOT / "models" / "action_model.pt").is_file():
    ROOT = ROOT.parent

MODEL_PATH = ROOT / "models" / "action_model.pt"
TEST_IMAGE = ROOT / "test" / "test.png"

if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"모델 없음: {MODEL_PATH}\n먼저 action_training_test.ipynb 로 학습하세요.")
if not TEST_IMAGE.is_file():
    raise FileNotFoundError(f"테스트 이미지 없음: {TEST_IMAGE}")

ACTION_KO = {
    "drawing": "그림그리기",
    "folding": "종이접기",
    "dancing": "춤추기",
}

print("ROOT:", ROOT)
print("MODEL:", MODEL_PATH)
print("IMAGE:", TEST_IMAGE)

In [ ]:
from IPython.display import Image, display

model = YOLO(str(MODEL_PATH))
results = model(str(TEST_IMAGE), verbose=False)
r = results[0]

class_id = int(r.probs.top1)
confidence = float(r.probs.top1conf)
action = r.names[class_id]

print("=== 예측 결과 ===")
print(f"클래스(영문): {action}")
print(f"클래스(한글): {ACTION_KO.get(action, action)}")
print(f"확률(top1):   {confidence:.4f} ({confidence * 100:.2f}%)")

print("\n=== 클래스별 확률 (전체) ===")
probs = r.probs.data.cpu().numpy()
for i, p in enumerate(probs):
    name = r.names[i]
    ko = ACTION_KO.get(name, name)
    bar = "#" * int(p * 40)
    print(f"  {ko:8s} ({name:8s})  {p:.4f}  ({p*100:5.2f}%)  {bar}")

display(Image(filename=str(TEST_IMAGE), width=320))